In [5]:



import inspect
import os
import CRUD_Python_Module

from CRUD_Python_Module import AnimalShelter

print("Imported from:")
print(CRUD_Python_Module.__file__)

print("Constructor expects:")
print(inspect.signature(AnimalShelter.__init__))

Imported from:
C:\Users\suare\Downloads\CS-340-main\CS-340-main\CS340Mod7\CRUD_Python_Module.py
Constructor expects:
(self, username, password, host, port, database, collection)


In [6]:
from CRUD_Python_Module import AnimalShelter
import pandas as pd
###########################
# Data Manipulation / Model
###########################

username = os.getenv("AAC_USERNAME")
password = os.getenv("AAC_PASSWORD")

host = os.getenv("AAC_HOST", "localhost")
port = int(os.getenv("AAC_PORT", "27017"))
database = os.getenv("AAC_DATABASE", "aac")
collection = os.getenv("AAC_COLLECTION", "animals")

shelter = AnimalShelter(username, password, host, port, database, collection)

# Test MongoDB connection through the CRUD object
print("MongoDB record count:")
print(shelter.collection.count_documents({}))

print("First record:")
print(shelter.collection.find_one())

# Load MongoDB data into dataframe
df = pd.DataFrame.from_records(shelter.read({}))

# Remove MongoDB ObjectId column if it exists
df.drop(columns=["_id"], inplace=True, errors="ignore")

print("Dataframe rows:", len(df))
print("Columns:")
print(df.columns)

MongoDB record count:
10000
First record:
{'_id': ObjectId('6a4f15a28dfe8e0bdedc1f93'), '': 1, 'age_upon_outcome': '3 years', 'animal_id': 'A746874', 'animal_type': 'Cat', 'breed': 'Domestic Shorthair Mix', 'color': 'Black/White', 'date_of_birth': datetime.datetime(2014, 4, 10, 0, 0), 'datetime': '2017-04-11 09:00:00', 'monthyear': '2017-04-11T09:00:00', 'name': '', 'outcome_subtype': 'SCRP', 'outcome_type': 'Transfer', 'sex_upon_outcome': 'Neutered Male', 'location_lat': 30.5066578739455, 'location_long': -97.3408780722188, 'age_upon_outcome_in_weeks': 156.767857142857}
Dataframe rows: 10000
Columns:
Index(['', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color',
       'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype',
       'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long',
       'age_upon_outcome_in_weeks'],
      dtype='str')


In [7]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports
import os
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
#JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


####  #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter



###########################
# Data Manipulation / Model
###########################
#update with your username and password and CRUD Python module name. NOTE: You will
# likely need more variables for your constructor to handle the hostname and port of the MongoDB
# server, and the database and collection names

username = os.getenv("AAC_USERNAME")
password = os.getenv("AAC_PASSWORD")

host = os.getenv("AAC_HOST", "localhost")
port = int(os.getenv("AAC_PORT", "27017"))
database = os.getenv("AAC_DATABASE", "aac")
collection = os.getenv("AAC_COLLECTION", "animals")

shelter = AnimalShelter(username, password, host, port, database, collection)

water_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ]
    },
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {
        "$gte": 26,
        "$lte": 156
    }
}

wilderness_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "German Shepherd", 
            "Alaskan",
            "Malamute", 
            "Old English Sheepdog", 
            "Siberian Husky", 
            "Rottweiler"
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {
        "$gte": 26,
        "$lte":156
    }
}

disaster_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Doberman Pinscher", 
            "German Shepherd",
            "Golden Retriever", 
            "Bloodhound", 
            "Rottweiler"
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {
        "$gte": 20,
        "$lte":300
    }
}

reset_query = {}


        




# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True, errors='ignore')

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('SNHU CS-340 Ricky Suarez Dashboard'))),
    
    html.Center(
        html.Img(
            src=app.get_asset_url('Grazioso_Salvare_Logo.png'),
            style={'width': '300px'}
        )
    ),
    html.Hr(),
    #Added Radio Option rather than buttons
    dcc.RadioItems(
        id='rescue-filter',
        options=[
            {'label': 'Water Rescue', 'value': 'water'},
            {'label': 'Wilderness Rescue', 'value': 'wilderness'},
            {'label': 'Disaster / Tracking', 'value': 'disaster'},
            {'label': 'Reset', 'value': 'reset'}
        ],
        value='reset',
        labelStyle={'display': 'block'}
    ),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        #Set up the features for your interactive data table to make it user-friendly for your client
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable='single',
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10
        ),
    html.Br(),
    html.Hr(),
    html.Div(
            id='map-id',
            className='col s12 m6',
            ),
    #Added secondary Bar Graph
    html.Br(),
    dcc.Graph(id='outcome-chart')
])

#############################################
# Interaction Between Components / Controller
#############################################
#This callback will highlight a row on the data table when the user selects it

@app.callback(
    Output('datatable-id', 'data'),
    Input('rescue-filter', 'value')
)
def update_table(filter_value):

    if filter_value == 'water':
        query = water_query
    elif filter_value == 'wilderness':
        query = wilderness_query
    elif filter_value == 'disaster':
        query = disaster_query
    else:
        query = reset_query

    filtered_df = pd.DataFrame.from_records(shelter.read(query))
    df.drop(columns=['_id'], inplace=True, errors='ignore')

    filtered_df.drop(
        columns=["_id"],
        inplace=True,
        errors="ignore"
    )

    return filtered_df.to_dict("records")




# Highlight a column when the user selects it.
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def update_styles(selected_columns):

    if not selected_columns:
        return []

    return [
        {
            "if": {
                "column_id": column
            },
            "background_color": "#D2F3FF"
        }
        for column in selected_columns
    ]


# Update the map for the selected data entry.
@app.callback(
    Output("map-id", "children"),
    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),
        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(viewData, index):

    # Prevent an error when the table contains no records.
    if not viewData:
        return html.Div(
            "No data available for map"
        )

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return html.Div(
            "No data available for map"
        )

    # Use the first record when no row has been selected.
    if not index:
        row = 0
    else:
        row = index[0]

    # Prevent an outdated selection from accessing an invalid row.
    if row < 0 or row >= len(dff):
        row = 0

    required_columns = [
        "location_lat",
        "location_long",
        "breed",
        "name"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in dff.columns
    ]

    if missing_columns:
        return html.Div(
            "The map cannot be displayed because required columns "
            "are missing."
        )

    latitude = pd.to_numeric(
        dff.loc[row, "location_lat"],
        errors="coerce"
    )

    longitude = pd.to_numeric(
        dff.loc[row, "location_long"],
        errors="coerce"
    )

    if pd.isna(latitude) or pd.isna(longitude):
        return html.Div(
            "The selected animal does not have valid coordinates."
        )

    animal_breed = dff.loc[row, "breed"]
    animal_name = dff.loc[row, "name"]

    if pd.isna(animal_breed):
        animal_breed = "Unknown breed"

    if pd.isna(animal_name):
        animal_name = "Unknown"

    return [
        dl.Map(
            style={
                "width": "1000px",
                "height": "500px"
            },

            center=[
                float(latitude),
                float(longitude)
            ],

            zoom=10,

            children=[
                dl.TileLayer(
                    id="base-layer-id"
                ),

                dl.Marker(
                    position=[
                        float(latitude),
                        float(longitude)
                    ],

                    children=[
                        dl.Tooltip(
                            str(animal_breed)
                        ),

                        dl.Popup([
                            html.H1(
                                "Animal Name"
                            ),

                            html.P(
                                str(animal_name)
                            )
                        ])
                    ]
                )
            ]
        )
    ]


@app.callback(
    Output("outcome-chart", "figure"),
    Input("datatable-id", "derived_virtual_data")
)
def update_outcome_chart(rows):

    if rows is None or len(rows) == 0:
        return px.bar(
            title="Outcome Type Frequency"
        )

    dff = pd.DataFrame(rows)

    if dff.empty or "outcome_type" not in dff.columns:
        return px.bar(
            title="Outcome Type Frequency"
        )

    outcome_counts = (
        dff["outcome_type"]
        .value_counts()
        .reset_index()
    )

    outcome_counts.columns = [
        "Outcome Type",
        "Count"
    ]

    fig = px.bar(
        outcome_counts,
        x="Outcome Type",
        y="Count",
        title="Outcome Type Frequency"
    )

    return fig


# Run the dashboard.
app.run(
    port=8055,
    debug=True
)